# CNN Project: From Scratch to State of the Art on CIFAR-10

**Estimated time:** 8–12 hours  |  **Total:** 100 points + 10 bonus  |  **Hardware:** Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU)

<font color='red'> **Make a copy of this notebook in your own Google Drive before you start (File → Save a copy in Drive), or your work will be lost.** </font>

---

## Project Overview

In this project you will build an image classifier for the **CIFAR-10** dataset: 60,000 small (32×32) colour photographs, each belonging to one of ten classes:

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|
| airplane | automobile | bird | cat | deer | dog | frog | horse | ship | truck |

Unlike a step-by-step tutorial, this project asks you to **predict before you run, justify every design decision, and analyse why your model fails**. You will move through the same journey a practitioner follows on a real problem:

| Problem | Topic | What you will learn | Points |
|---|---|---|---|
| 0 | Data understanding & preprocessing | Splits, class balance, memory-efficient pipelines | 8 |
| 1 | Baseline CNN | Building from a spec, shapes and parameters by hand | 17 |
| 2 | Architecture experiments | Global pooling, Batch Normalization, deeper blocks, Dropout | 20 |
| 3 | Augmentation & training control | Label-preserving augmentation, callbacks, learning-rate schedules | 15 |
| 4 | Transfer learning & fine-tuning | MobileNetV2, preprocessing traps, two-phase fine-tuning | 20 |
| 5 | Error analysis & interpretability | Confusion analysis, failure galleries, **Grad-CAM from scratch** | 12 |
| 6 | Data-efficiency study | When does transfer learning matter most? | 8 |
| 7 | **Bonus challenge** | Beat an accuracy target under a parameter budget | +10 |

## How this notebook works

You will see four kinds of task cells:

* `### YOUR CODE HERE ###` — write code.
* 🔮 **Predict first** — write down a number *before* you run the code. A checker tells you whether you were right. Your prediction is graded on your *reasoning*, not only on being correct, so show your work.
* ✍️ <font color='red'>**Your Answer**</font> — written explanation. Aim for 3–6 sentences that cite *your own* numbers and plots.
* ✅ **Self-check** cells — run them; they catch common mistakes early. A passing self-check does not guarantee full marks.

## Rules of the game

1. **The test set is for reporting, not for decisions.** Choose architectures, epochs and hyperparameters using the *validation* set only. Using test accuracy to make choices is a form of data leakage, and in Problem 7 it is disqualifying.
2. **Keep the seed fixed** so your results are reproducible.
3. **Cite your evidence.** "Model B is better" is not an answer; "Model B reaches 78.4% validation accuracy versus 71.2% for Model A, and its train–validation gap shrinks from 19 to 6 points" is.
4. You may use any documentation, but every line of code you submit must be code you can explain.

# Setup

Run the cells below. The first cell confirms that you are on a GPU; training without one will be 10–30× slower.

In [ ]:
import os, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from sklearn import metrics
from sklearn.model_selection import train_test_split
from IPython.display import display

# Fix all random seeds (Python, NumPy, TensorFlow) so results are reproducible
SEED = 42
keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", gpus if gpus else "NO GPU - switch runtime to GPU before continuing!")

In [ ]:
# Global training settings. You may lower EPOCHS while debugging,
# but your final submitted runs must use the values below unless a problem says otherwise.
BATCH_SIZE = 64
EPOCHS = 20

### Helper utilities (provided)

These helpers are given to you so that every experiment is evaluated the same way. **Read them before using them** — several questions later in the project assume you understand what they compute.

* `plot_history(history)` plots accuracy **and** loss for training and validation.
* `record(name, model, history)` evaluates a model and saves its key numbers to the `RESULTS` table.
* `show_results()` displays every experiment you have recorded, side by side.
* `check(label, value, expected)` is a small self-check helper.

In [ ]:
RESULTS = {}

def plot_history(history, title=""):
    # Plot training/validation accuracy and loss side by side.
    h = history.history
    epochs = range(1, len(h["accuracy"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(epochs, h["accuracy"], "o-", label="train")
    axes[0].plot(epochs, h["val_accuracy"], "s-", label="validation")
    axes[0].set(title=f"{title} accuracy", xlabel="epoch", ylabel="accuracy")
    axes[1].plot(epochs, h["loss"], "o-", label="train")
    axes[1].plot(epochs, h["val_loss"], "s-", label="validation")
    axes[1].set(title=f"{title} loss", xlabel="epoch", ylabel="loss")
    for ax in axes:
        ax.legend(); ax.grid(alpha=0.3)
    plt.show()

def record(name, model, history):
    # Evaluate on validation and test sets and store the summary in RESULTS.
    h = history.history
    best_epoch = int(np.argmax(h["val_accuracy"]))
    _, test_acc = model.evaluate(x_test, y_test, verbose=0)
    trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    RESULTS[name] = {
        "total_params": model.count_params(),
        "trainable_params": trainable,
        "epochs_run": len(h["accuracy"]),
        "best_val_acc": round(h["val_accuracy"][best_epoch], 4),
        "train_acc@best": round(h["accuracy"][best_epoch], 4),
        "gap@best": round(h["accuracy"][best_epoch] - h["val_accuracy"][best_epoch], 4),
        "test_acc": round(test_acc, 4),
    }
    print(f"[{name}] best val acc = {RESULTS[name]['best_val_acc']:.4f} | test acc = {test_acc:.4f}")

def show_results():
    display(pd.DataFrame(RESULTS).T)

def check(label, value, expected):
    if value is None:
        print(f"⚠️  {label}: you have not filled this in yet.")
    elif value == expected:
        print(f"✅ {label}: correct ({value:,})")
    else:
        print(f"❌ {label}: you predicted {value:,} but the model has {expected:,}. Find the error in your reasoning and explain it in your answer.")

# Problem 0: Data Understanding & Preprocessing [8 points]

Keras ships CIFAR-10 directly, so no download link is needed.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train_full = y_train_full.squeeze()   # shape (50000, 1) -> (50000,)
y_test = y_test.squeeze()

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

print("train:", x_train_full.shape, x_train_full.dtype, "| test:", x_test.shape)
print("pixel range:", x_train_full.min(), "to", x_train_full.max())

## Part (a): Explore the data [2 points]

1. Plot a 4×8 grid of random training images with their class names as titles.
2. Plot a bar chart of how many images belong to each class in the training set.

In [ ]:
### YOUR CODE HERE ###

✍️ Is the dataset balanced? Why does the answer matter for how you interpret *accuracy* as a metric? Give one example of a dataset where accuracy would be a misleading metric.

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

## Part (b): A proper validation split [3 points]

In many tutorials, `model.fit(..., validation_split=0.2)` is used. That option simply takes the **last** 20% of the array as validation data, without shuffling.

Instead, create an explicit **stratified** split with `train_test_split`: hold out **5,000** images for validation, use `stratify=y_train_full` and `random_state=SEED`. Store the results in `x_train, x_val, y_train, y_val`.

In [ ]:
### YOUR CODE HERE ###
x_train, x_val, y_train, y_val = None, None, None, None

In [ ]:
# ✅ Self-check
assert x_train.shape == (45000, 32, 32, 3), f"x_train has shape {x_train.shape}"
assert x_val.shape == (5000, 32, 32, 3), f"x_val has shape {x_val.shape}"
assert np.all(np.bincount(y_val) == 500), "Validation set is not stratified (each class should have exactly 500 images)"
print("✅ Split looks correct")

✍️ Answer both:

1. Describe a concrete situation where `validation_split=0.2` (last 20%, unshuffled) would give a badly misleading validation score.
2. Why do we need a validation set at all when we already have a test set?

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

## Part (c): Labels and memory [3 points]

Notice two choices we make in this project that differ from many tutorials:

* **Integer labels, not one-hot.** `y_train` contains integers 0–9, and we will use the `sparse_categorical_crossentropy` loss.
* **Images stay as `uint8`.** We will **not** divide by 255 in NumPy. Instead, every model begins with a `Rescaling` layer that converts pixels to floats inside the network.

Compute (in code) how many megabytes `x_train` occupies as `uint8`, and how many it would occupy as `float32`.

In [ ]:
### YOUR CODE HERE ###

✍️ Answer both:

1. Mathematically, is `sparse_categorical_crossentropy` with integer labels different from `categorical_crossentropy` with one-hot labels? When would you prefer one over the other?
2. Give two advantages of doing rescaling *inside* the model (as a layer) rather than in NumPy beforehand. Hint: think about memory and about what happens when the model is deployed.

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

# Problem 1: Baseline CNN [17 points]

## Part (a): Build the model from a specification [5 points]

Build `baseline` using the Keras **Functional API** exactly as specified below. All convolutions use a 3×3 kernel, `padding="same"` and ReLU activation. All pooling layers are 2×2 max pooling.

| # | Layer | Configuration |
|---|---|---|
| 1 | Input | shape (32, 32, 3) |
| 2 | Rescaling | scale 1/255 |
| 3 | Conv2D | 32 filters |
| 4 | MaxPooling2D | 2×2 |
| 5 | Conv2D | 64 filters |
| 6 | MaxPooling2D | 2×2 |
| 7 | Conv2D | 128 filters |
| 8 | MaxPooling2D | 2×2 |
| 9 | Flatten | |
| 10 | Dense | 128 units, ReLU |
| 11 | Dense (output) | 10 units, *choose the correct activation* |

The finished model has **356,810** parameters.

In [ ]:
inputs = keras.Input(shape=(32, 32, 3), name="input")

### YOUR CODE HERE ###


### YOUR CODE HERE ###

baseline = keras.Model(inputs, outputs, name="baseline")
baseline.summary()

In [ ]:
# ✅ Self-check
assert baseline.count_params() == 356_810, f"Expected 356,810 parameters, found {baseline.count_params():,}"
assert baseline.output_shape == (None, 10)
print("✅ Architecture matches the specification")

## Part (b): Shapes and parameters by hand [6 points]

Without looking at `baseline.summary()`, fill in the table below **by hand**. For each parameter count, show the formula (for example, `896 = 3·3·3·32 + 32`).

| Layer | Output shape | # Parameters | Formula |
|---|---|---|---|
| Conv2D (32) | | | |
| MaxPooling2D | | | |
| Conv2D (64) | | | |
| MaxPooling2D | | | |
| Conv2D (128) | | | |
| MaxPooling2D | | | |
| Flatten | | | |
| Dense (128) | | | |
| Dense (10) | | | |

Then answer:

1. Which single layer holds the most parameters, and what fraction of the total is it? What does this suggest about where the model might be overfitting?
2. If we replaced the whole convolutional part with a plain MLP — `Flatten` the raw 32×32×3 image and feed it directly into a `Dense(128)` layer — how many parameters would that one Dense layer have? Compare it with the parameter count of all three Conv2D layers combined, and explain in terms of **weight sharing** and **local connectivity** why CNNs suit images better.
3. How would the output shape of the first Conv2D change if we used `padding="valid"` instead of `"same"`?

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

## Part (c): Train and evaluate [6 points]

Compile `baseline` with the `adam` optimizer, `sparse_categorical_crossentropy` loss and the `accuracy` metric. Train it on `(x_train, y_train)` for `EPOCHS` epochs with `BATCH_SIZE`, passing `validation_data=(x_val, y_val)`. Store the result in `baseline_history`.

In [ ]:
### YOUR CODE HERE ###
baseline.compile(

)

baseline_history = baseline.fit(

)

In [ ]:
plot_history(baseline_history, "Baseline")
record("baseline", baseline, baseline_history)
show_results()

✍️ Answer all:

1. How many batches (steps) are there per epoch? Show the calculation. Why is it not a whole number before rounding, and what does Keras do with the leftover images?
2. In your own words, explain the relationship between a **batch**, an **epoch** and one **gradient update**.
3. Using **both** plots (accuracy *and* loss), identify the epoch at which the model starts to overfit. Explain why the validation *loss* often reveals overfitting earlier than validation *accuracy* does.

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

# Problem 2: Architecture Experiments [20 points]

You will now change the architecture **one idea at a time**, so that each change can be attributed to its cause. This is the discipline of a *controlled experiment*.

Every model in this problem must be compiled and trained exactly like `baseline` (same optimizer, loss, epochs, batch size and validation data). Write a helper `compile_and_fit(model)` once and reuse it.

In [ ]:
def compile_and_fit(model, epochs=EPOCHS, callbacks=None, optimizer="adam"):
    ### YOUR CODE HERE ###
    # Compile the model like baseline and train it on (x_train, y_train),
    # validating on (x_val, y_val). Return the History object.
    pass

## Part (a): Global Average Pooling [5 points]

Build `gap_model`: identical to `baseline`, except replace `Flatten` with `GlobalAveragePooling2D`.

🔮 **Predict first.** Before building it, compute the total number of parameters of `gap_model` by hand and write it in `pred_gap_params`.

In [ ]:
pred_gap_params = None   # 🔮 your prediction, as an integer

inputs = keras.Input(shape=(32, 32, 3))
### YOUR CODE HERE ###


gap_model = keras.Model(inputs, outputs, name="gap_model")
check("gap_model parameters", pred_gap_params, gap_model.count_params())

In [ ]:
gap_history = compile_and_fit(gap_model)
plot_history(gap_history, "GAP")
record("gap_model", gap_model, gap_history)
show_results()

✍️ Show your parameter calculation. Then: by what factor did the parameter count shrink? Did accuracy go up or down, and did the train–validation gap change? Explain *what information* `GlobalAveragePooling2D` throws away compared with `Flatten`, and why that can act as a regulariser.

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

## Part (b): Batch Normalization [6 points]

Build `bn_model`: identical to `gap_model`, except every convolution becomes the pattern

`Conv2D (no activation) → BatchNormalization → ReLU`

🔮 **Predict first.** Batch Normalization has both *trainable* and *non-trainable* parameters. Predict the total, trainable, and non-trainable parameter counts of `bn_model`.

In [ ]:
pred_bn_total = None          # 🔮
pred_bn_trainable = None      # 🔮
pred_bn_non_trainable = None  # 🔮

inputs = keras.Input(shape=(32, 32, 3))
### YOUR CODE HERE ###


bn_model = keras.Model(inputs, outputs, name="bn_model")

trainable = int(sum(np.prod(w.shape) for w in bn_model.trainable_weights))
non_trainable = int(sum(np.prod(w.shape) for w in bn_model.non_trainable_weights))
check("bn_model total", pred_bn_total, bn_model.count_params())
check("bn_model trainable", pred_bn_trainable, trainable)
check("bn_model non-trainable", pred_bn_non_trainable, non_trainable)

In [ ]:
bn_history = compile_and_fit(bn_model)
plot_history(bn_history, "BatchNorm")
record("bn_model", bn_model, bn_history)
show_results()

✍️ Answer all:

1. Name the four parameter vectors that each BatchNormalization layer stores. Which two are learned by gradient descent, and how are the other two updated?
2. BatchNorm behaves differently during `fit()` and during `evaluate()` / `predict()`. Describe the difference precisely.
3. Why is the bias term of a convolution that is immediately followed by BatchNorm redundant? (Optional: rebuild the model with `use_bias=False` and confirm how many parameters you save.)
4. Compare `bn_model` with `gap_model`: what changed in the *speed of learning* in the first few epochs, and in final accuracy?

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

## Part (c): Deeper blocks with Dropout [5 points]

Real CNNs stack several convolutions before each pooling step. Build `deep_model` with three blocks. Each block is:

`[Conv2D → BatchNorm → ReLU] × 2 → MaxPooling2D(2×2) → Dropout(0.25)`

using 32, 64 and 128 filters in blocks 1, 2 and 3. After the blocks: `GlobalAveragePooling2D → Dense(128, ReLU) → Dropout(0.5) → Dense(10)`.

**Requirement:** write a reusable Python function `conv_block(x, filters, name)` that returns the output tensor of one block. You will reuse it in Problem 3.

🔮 **Predict first:** the total parameter count of `deep_model`.

In [ ]:
def conv_block(x, filters, name):
    ### YOUR CODE HERE ###
    pass

pred_deep_params = None   # 🔮

inputs = keras.Input(shape=(32, 32, 3))
### YOUR CODE HERE ###


deep_model = keras.Model(inputs, outputs, name="deep_model")
check("deep_model parameters", pred_deep_params, deep_model.count_params())

In [ ]:
deep_history = compile_and_fit(deep_model)
plot_history(deep_history, "Deep + Dropout")
record("deep_model", deep_model, deep_history)
show_results()

✍️ **Synthesis question.** Using the `RESULTS` table, write a short paragraph (6–10 sentences) comparing `baseline`, `gap_model`, `bn_model` and `deep_model`.

* Which change produced the largest gain in validation accuracy?
* Is there a relationship between parameter count and accuracy in your results? Is "more parameters → better" true here?
* Dropout is active during training but not during evaluation. How does this explain why, for `deep_model`, training accuracy can even be *lower* than validation accuracy in early epochs?

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

# Problem 3: Data Augmentation & Training Control [15 points]

## Part (a): Choosing label-preserving augmentations [4 points]

Data augmentation only helps if the transformed image would *still be labelled the same way* by a human. Consider these candidate Keras layers:

`RandomFlip("horizontal")`, `RandomFlip("vertical")`, `RandomRotation(0.05)`, `RandomRotation(0.5)`, `RandomTranslation(0.1, 0.1)`, `RandomZoom(0.1)`, `RandomContrast(0.2)`

✍️ For **each** candidate, decide whether it is appropriate for CIFAR-10 and justify it in one line. Would your answer for `RandomFlip("horizontal")` change if the dataset were handwritten digits (MNIST)? Why?

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

Now create a `keras.Sequential` model called `augment` containing the augmentations you selected, and visualise them: show one training image and 8 augmented versions of it.

**Important:** augmentation layers are only active when `training=True`. Call `augment(images, training=True)` for the visualisation.

In [ ]:
### YOUR CODE HERE ###
augment = keras.Sequential([
    # your chosen augmentation layers
], name="augment")

# visualise 1 original + 8 augmented versions of x_train[0]

In [ ]:
# ✅ Self-check: augmentation must be OFF at inference time
sample = x_train[:4].astype("float32")
assert np.allclose(augment(sample, training=False), sample), \
    "Your augmentation changes images at inference time - it should not."
assert not np.allclose(augment(sample, training=True), sample), \
    "Your augmentation does nothing during training - did you add any layers?"
print("✅ Augmentation is active in training and inactive at inference")

## Part (b): Augmented model with callbacks [6 points]

Build `aug_model` = `augment` placed **immediately after the Input layer**, followed by the same architecture as `deep_model` (reuse `conv_block`).

Augmented models learn more slowly, so train for **40 epochs** and use these callbacks:

* `EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True)`
* `ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)`
* A small custom callback (or the `History` object) that lets you plot the **learning rate per epoch**.

In [ ]:
inputs = keras.Input(shape=(32, 32, 3))
### YOUR CODE HERE ###


aug_model = keras.Model(inputs, outputs, name="aug_model")

callbacks = [
    ### YOUR CODE HERE ###
]

aug_history = compile_and_fit(aug_model, epochs=40, callbacks=callbacks)

In [ ]:
plot_history(aug_history, "Augmented")
record("aug_model", aug_model, aug_history)
show_results()

### YOUR CODE HERE ###  plot the learning rate across epochs

✍️ Answer all:

1. Did augmentation reduce overfitting? Quote the train–validation gap for `deep_model` and `aug_model`.
2. Mark on your learning-rate plot where `ReduceLROnPlateau` fired. What happened to the validation loss immediately afterwards, and why?
3. `restore_best_weights=True` means the final model is the one from the best validation epoch. Why is it legitimate to choose the epoch with validation data, but **not** with test data?

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

## Part (c): Mini ablation study [5 points]

An *ablation* removes one component at a time to measure how much it contributes. Choose **two** of your augmentations and train two extra models, each with **one** of them removed (keep everything else identical). Record them as `aug_minus_<name>`.

To save time, you may use 25 epochs for these runs, but then also re-run the full `aug_model` with 25 epochs so the comparison is fair.

In [ ]:
### YOUR CODE HERE ###

✍️ Present a small table of the ablation results. Which augmentation contributes the most? Is the difference larger than what you would expect from random run-to-run variation? (Hint: how could you estimate that variation?)

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

# Problem 4: Transfer Learning & Fine-Tuning [20 points]

We now reuse **MobileNetV2**, a network pre-trained on 1.2 million ImageNet photos. You will compare three strategies:

1. **Feature extraction:** freeze the whole pre-trained base and train only a new classification head.
2. **Partial fine-tuning:** after step 1, unfreeze the *top* part of the base and continue training with a small learning rate.
3. **Full fine-tuning:** unfreeze the whole base.

## Part (a): Input size and preprocessing — the hidden traps [5 points]

Two details make or break transfer learning:

* **Resolution.** ImageNet weights for MobileNetV2 were trained on much larger images. CIFAR images are 32×32, so we upscale them inside the model with a `Resizing(96, 96)` layer.
* **Preprocessing.** A pre-trained network expects inputs in *exactly* the numeric range it was trained on. Run the cell below to see what MobileNetV2 expects.

In [ ]:
probe = np.array([[0.0, 127.5, 255.0]], dtype="float32")
print("mobilenet_v2.preprocess_input maps", probe, "->",
      keras.applications.mobilenet_v2.preprocess_input(probe.copy()))

Write a single `Rescaling` layer (choose `scale` and `offset`) that reproduces this mapping exactly, and verify it numerically.

In [ ]:
### YOUR CODE HERE ###
mobilenet_rescale = None   # keras.layers.Rescaling(scale=..., offset=...)

assert np.allclose(mobilenet_rescale(probe),
                   keras.applications.mobilenet_v2.preprocess_input(probe.copy())), "Mapping does not match"
print("✅ Preprocessing matches MobileNetV2")

✍️ Answer both:

1. Suppose someone used `Rescaling(1/255)` (range [0, 1]) in front of MobileNetV2 instead of the correct mapping. What would the first convolution layer "see" compared with what it was trained on, and how would you expect this to affect frozen-base accuracy?
2. Why do we upscale to 96×96 instead of feeding 32×32 images? What is the cost of upscaling?

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

## Part (b): Feature extraction (frozen base) [5 points]

Build `transfer_model`:

`Input → augment → Resizing(96, 96) → mobilenet_rescale → base (frozen) → GlobalAveragePooling2D → Dropout(0.3) → Dense(10)`

Requirements:

* Load the base with `keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(96, 96, 3))` and set `base.trainable = False`.
* Call the base as `base(x, training=False)`. (You will explain why in Part (d).)
* Compile with `Adam(1e-3)` and train for **10 epochs** with `EarlyStopping` on `val_accuracy`.

In [ ]:
### YOUR CODE HERE ###
base = None

inputs = keras.Input(shape=(32, 32, 3))


transfer_model = keras.Model(inputs, outputs, name="transfer_frozen")
print("Layers in base:", len(base.layers))

In [ ]:
### YOUR CODE HERE ###  compile and train

transfer_history = None

In [ ]:
plot_history(transfer_history, "Frozen MobileNetV2")
record("transfer_frozen", transfer_model, transfer_history)
show_results()

## Part (c): Partial and full fine-tuning [5 points]

**Partial fine-tuning.** Continue from the trained `transfer_model` (do *not* start over):

1. Set `base.trainable = True`, then freeze every layer of `base` **before index 100** again.
2. **Re-compile** with a much smaller learning rate, `Adam(1e-5)`.
3. Train for 10 more epochs. Pass `initial_epoch` so the epoch numbering continues.

Print how many layers of `base` are trainable before training.

**Full fine-tuning.** Rebuild a fresh copy of the frozen-base model, train its head for 10 epochs exactly as in Part (b), then unfreeze **all** of `base`, re-compile with `Adam(1e-5)` and train for 10 more epochs. Record both models.

In [ ]:
### YOUR CODE HERE ###  partial fine-tuning

partial_history = None

In [ ]:
### YOUR CODE HERE ###  full fine-tuning (fresh model, head first, then unfreeze everything)

full_history = None

In [ ]:
### YOUR CODE HERE ###  plot both histories and record them as "transfer_partial" and "transfer_full"

show_results()

## Part (d): Analysis [5 points]

✍️ Answer all, citing your `RESULTS` table:

1. Why must the head be trained (with the base frozen) **before** unfreezing the base? What would large gradients from a randomly initialised head do to the pre-trained weights?
2. Why must you **re-compile** after changing `trainable`?
3. The base contains BatchNormalization layers. Why do we keep calling it with `training=False`, even during fine-tuning?
4. Lower layers of a CNN learn generic features (edges, colours, textures), upper layers learn task-specific ones. Use this idea to explain why we unfreeze the *top* layers in partial fine-tuning, not the bottom ones.
5. Compare the best transfer model with your best from-scratch model on accuracy, number of parameters, and training time per epoch. When would you choose the from-scratch model in a real application?

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

# Problem 5: Error Analysis & Interpretability [12 points]

Accuracy tells you *how often* a model fails. Error analysis tells you *how* and *why*. Pick your **best model by validation accuracy** and assign it to `best_model`.

In [ ]:
best_model = None   ### YOUR CODE HERE ###

## Part (a): Confusion analysis [4 points]

1. Compute predictions on the test set and plot a **row-normalised** confusion matrix heatmap with class names.
2. Print the per-class precision and recall (`metrics.classification_report`).
3. Write code that automatically lists the **five most confused pairs** (true class → predicted class), excluding the diagonal, with their counts.

In [ ]:
### YOUR CODE HERE ###

✍️ Which classes are hardest? Are the most confused pairs *semantically* similar (for example, two animals) or *visually* similar (similar colours or backgrounds)? Propose one concrete change to the data or the model that might reduce the top confusion.

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

## Part (b): Failure gallery [3 points]

Display the **16 test images the model got wrong with the highest confidence**. For each, show the true label, the predicted label and the predicted probability.

In [ ]:
### YOUR CODE HERE ###

✍️ Look carefully at the gallery. Sort the errors into categories, for example: *genuinely ambiguous*, *label looks wrong*, *unusual viewpoint*, *misleading background*. Roughly how many fall into each? What does a highly confident mistake tell you about whether the model's probabilities are *calibrated*?

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

## Part (c): Grad-CAM from scratch [5 points]

**Grad-CAM** highlights the image regions that most influenced a prediction. The idea:

1. Take the feature maps $A^k$ of the last convolutional stage (shape $H \times W \times K$).
2. Compute the gradient of the class score $y^c$ with respect to each feature map.
3. Average those gradients over the spatial dimensions to get one importance weight per channel: $\alpha_k = \frac{1}{HW}\sum_{i,j} \frac{\partial y^c}{\partial A^k_{ij}}$.
4. The heatmap is $\text{ReLU}\left(\sum_k \alpha_k A^k\right)$, normalised to $[0, 1]$.

Use one of your **from-scratch** models (for example `aug_model`). In that model, the final ReLU output of the last convolutional block must have a layer name you know. Use `model.summary()` to find it, or rebuild the block with `name="last_conv"` on that activation.

Complete the four TODOs below.

In [ ]:
def make_gradcam_heatmap(img, model, last_conv_layer_name, class_index=None):
    # img: a single uint8 image of shape (32, 32, 3)
    grad_model = keras.Model(model.inputs,
                             [model.get_layer(last_conv_layer_name).output, model.output])
    img_batch = tf.convert_to_tensor(img[None, ...], dtype=tf.float32)

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch, training=False)
        if class_index is None:
            class_index = int(tf.argmax(preds[0]))
        class_score = preds[:, class_index]

    # TODO 1: gradient of class_score with respect to conv_out
    grads = None
    # TODO 2: average grads over batch, height and width -> one weight per channel, shape (K,)
    pooled_grads = None
    # TODO 3: weighted sum of the channels of conv_out[0] -> shape (H, W)
    heatmap = None
    # TODO 4: apply ReLU and normalise to [0, 1] (guard against division by zero)
    heatmap = None

    return heatmap.numpy(), class_index

In [ ]:
def show_gradcam(indices, model, layer_name):
    fig, axes = plt.subplots(2, len(indices), figsize=(2.4 * len(indices), 5))
    for col, i in enumerate(indices):
        heatmap, c = make_gradcam_heatmap(x_test[i], model, layer_name)
        heatmap = tf.image.resize(heatmap[..., None], (32, 32)).numpy().squeeze()
        axes[0, col].imshow(x_test[i]); axes[0, col].axis("off")
        axes[0, col].set_title(f"true: {CLASS_NAMES[y_test[i]]}", fontsize=9)
        axes[1, col].imshow(x_test[i]); axes[1, col].imshow(heatmap, cmap="jet", alpha=0.45)
        axes[1, col].axis("off"); axes[1, col].set_title(f"pred: {CLASS_NAMES[c]}", fontsize=9)
    plt.tight_layout(); plt.show()

### YOUR CODE HERE ###
# Call show_gradcam on 6 correctly classified and 6 misclassified test images.

✍️ Does the model look at the object or at the background? Find at least one example where the heatmap reveals that the model was "right for the wrong reason" or wrong for an understandable reason. Why is the Grad-CAM heatmap so coarse (blocky) for a 32×32 input, and how does the architecture determine its resolution?

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

# Problem 6: Data-Efficiency Study [8 points]

How much labelled data do you really need? Train two model families on **stratified subsets** of the training set containing **1,000**, **5,000** and **all 45,000** images:

* **Scratch:** your `aug_model` architecture, trained from random initialisation (up to 40 epochs, early stopping).
* **Transfer:** frozen MobileNetV2 + head, as in Problem 4(b).

Always evaluate on the same `x_val` / `x_test`. You may reuse your earlier full-data results instead of retraining them. Build fresh models for every run; never continue training a model that has already seen other data.

In [ ]:
def stratified_subset(n):
    # Return (x, y) containing n training images with equal class proportions.
    ### YOUR CODE HERE ###
    pass

### YOUR CODE HERE ###  run the experiments and collect test accuracies in a DataFrame

In [ ]:
### YOUR CODE HERE ###
# Plot test accuracy (y-axis) against training-set size (x-axis, log scale), one line per model family.

✍️ At which data size is the advantage of transfer learning largest? Does the gap close as data grows? Write a recommendation (3–5 sentences) for a company that has only 2,000 labelled images of its own products: which approach should it use first and why?

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

# Problem 7 (Bonus): The Parameter-Budget Challenge [+10 points]

Design your own CNN **from scratch** (no pre-trained weights) with **at most 250,000 parameters** and achieve the highest test accuracy you can.

| Tier | Test accuracy | Bonus |
|---|---|---|
| 🥉 Bronze | ≥ 82% | +4 |
| 🥈 Silver | ≥ 86% | +7 |
| 🥇 Gold | ≥ 89% | +10 |

**Rules**

* All design choices must be made on the **validation set**. Evaluate on the test set **exactly once**, at the very end, and report that number. Multiple test evaluations are disqualifying.
* Training must finish in under **30 minutes** on a Colab T4 GPU.
* Ideas worth exploring: residual (skip) connections, `SeparableConv2D`, wider-but-shallower vs narrower-but-deeper designs, label smoothing, a cosine learning-rate schedule, `AdamW` with weight decay.

In [ ]:
### YOUR CODE HERE ###
challenge_model = None

In [ ]:
# ✅ Self-check (run before training)
assert challenge_model.count_params() <= 250_000, f"Too many parameters: {challenge_model.count_params():,}"
print(f"✅ {challenge_model.count_params():,} parameters - within budget")

✍️ Document your design process: list each idea you tried, its validation accuracy, and whether you kept it. Explain the one idea that helped most and *why* you think it worked.

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

# Final Reflection & Submission

## Final results table

In [ ]:
show_results()

✍️ **Executive summary (8–12 sentences).** Imagine you are presenting to a manager who does not know deep learning. Summarise what you built, which approach worked best, what the model still gets wrong, and what you would do with two more weeks. Avoid jargon or explain it when used.

<font color='red'>**Your Answer.**</font>

_Replace this text with your answer._

---

## Submission checklist

- [ ] Every code cell runs top-to-bottom with **Runtime → Restart and run all**, without errors.
- [ ] All 🔮 prediction variables are filled in and their reasoning is shown in the written answers.
- [ ] All ✅ self-checks pass.
- [ ] Every ✍️ answer cites specific numbers from *your* results.
- [ ] The `RESULTS` table is complete.
- [ ] Submit the `.ipynb` **with outputs** and a PDF export (File → Print → Save as PDF).

## Grading rubric

| Criterion | Weight | Excellent work |
|---|---|---|
| Correctness of code | 40% | Models match specifications; self-checks pass; experiments are controlled. |
| Quality of reasoning | 35% | Answers explain *why*, cite evidence, and connect results to concepts. |
| Predictions | 10% | Hand calculations are shown step by step; errors are diagnosed honestly. |
| Experimental discipline | 10% | Decisions use validation data only; comparisons are fair. |
| Communication | 5% | Plots are labelled; tables are tidy; the executive summary is clear. |